# SpatialData Prototype: `SLIDE-0329_crop_2048`

This notebook prototypes SpatialData ingestion using the existing cropped outputs at `/mnt/c/Analysis/Data_prototype/SLIDE-0329_crop_2048`.

Scope for this notebook:
- load the existing full merged OME-TIFF into SpatialData,
- compare multiple image import routes,
- load the existing whole-cell mask into the same object,
- optionally load the nuclear mask,
- verify image and labels align in the same coordinate system.

Out of scope for this notebook:
- rerunning merge, InstanSeg, export, or Nimbus,
- table import,
- channel aggregation,
- productionizing the workflow into the pipeline.


In [ ]:
from __future__ import annotations

from importlib import import_module
from pathlib import Path
import json
import os
import re

os.environ.setdefault("NUMBA_CACHE_DIR", "/tmp/numba_cache")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig")

CROP_DIR = Path("/mnt/c/Analysis/Data_prototype/SLIDE-0329_crop_2048")
OUTPUT_DIR = CROP_DIR / "outputs"
FULL_MERGE_PATH = OUTPUT_DIR / "SLIDE-0329_crop_2048_full_merge.ome.tif"
SEGMENT_MERGE_PATH = OUTPUT_DIR / "SLIDE-0329_crop_2048_segment_merge.ome.tif"
CELL_MASK_PATH = OUTPUT_DIR / "masks" / "SLIDE-0329_crop_2048_whole_cell.tiff"
NUCLEAR_MASK_PATH = OUTPUT_DIR / "masks" / "SLIDE-0329_crop_2048_nuclear.tiff"
CHANNEL_MAP_PATH = OUTPUT_DIR / "channel_map.generated.json"

for path in [CROP_DIR, OUTPUT_DIR, FULL_MERGE_PATH, CELL_MASK_PATH, CHANNEL_MAP_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

print("Crop dir:", CROP_DIR)
print("Full merge:", FULL_MERGE_PATH)
print("Cell mask:", CELL_MASK_PATH)
print("Nuclear mask exists:", NUCLEAR_MASK_PATH.exists())


In [ ]:
def require_module(name: str, *, package_name: str | None = None):
    package_name = package_name or name
    try:
        module = import_module(name)
    except Exception as exc:
        raise ImportError(
            f"Missing required module '{name}'. Install '{package_name}' in the active environment before running this notebook."
        ) from exc
    version = getattr(module, "__version__", "<no __version__>")
    print(f"{name}: {version}")
    return module


spatialdata = require_module("spatialdata")
spatialdata_io = require_module("spatialdata_io", package_name="spatialdata-io")
sopa = require_module("sopa")
tifffile = require_module("tifffile")
np = require_module("numpy")
pd = require_module("pandas")
xr = require_module("xarray")
ome_types = require_module("ome_types", package_name="ome-types")

try:
    matplotlib = require_module("matplotlib")
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False
    plt = None
    print("matplotlib: not available; overlay plotting cells will be skipped")

from spatialdata import SpatialData
from spatialdata.models import Image2DModel, Labels2DModel

DataTree = xr.DataTree


In [ ]:
with CHANNEL_MAP_PATH.open("r", encoding="utf-8") as handle:
    channel_map = json.load(handle)

channel_aliases = [entry["alias"] for entry in channel_map]
channel_source_names = [entry.get("nimbus_name", Path(entry["path"]).stem) for entry in channel_map]

print(f"channel_map entries: {len(channel_map)}")
print("first aliases:", channel_aliases[:8])
print("first source names:", channel_source_names[:4])


In [ ]:
def inspect_ome_tiff(path: Path) -> dict:
    with tifffile.TiffFile(path) as handle:
        series = handle.series[0]
        ome_xml = handle.ome_metadata
        axes = series.axes
        shape = tuple(series.shape)
        dtype = str(series.dtype)
        level_shapes = [tuple(level.shape) for level in series.levels]
        page0 = series.levels[0].pages[0]

    metadata = ome_types.from_xml(ome_xml) if ome_xml else None
    pixels = metadata.images[0].pixels if metadata is not None else None
    ome_channel_names = [channel.name for channel in pixels.channels] if pixels is not None else []

    return {
        "path": str(path),
        "axes": axes,
        "shape": shape,
        "dtype": dtype,
        "level_shapes": level_shapes,
        "is_tiled": bool(page0.is_tiled),
        "tile": (page0.tilelength, page0.tilewidth) if page0.is_tiled else None,
        "ome_channel_names": ome_channel_names,
        "physical_size_x": None if pixels is None else pixels.physical_size_x,
        "physical_size_y": None if pixels is None else pixels.physical_size_y,
        "physical_size_x_unit": None if pixels is None else str(pixels.physical_size_x_unit),
        "physical_size_y_unit": None if pixels is None else str(pixels.physical_size_y_unit),
    }


full_merge_info = inspect_ome_tiff(FULL_MERGE_PATH)
segment_merge_info = inspect_ome_tiff(SEGMENT_MERGE_PATH)

print("full merge axes:", full_merge_info["axes"])
print("full merge shape:", full_merge_info["shape"])
print("full merge dtype:", full_merge_info["dtype"])
print("full merge levels:", full_merge_info["level_shapes"])
print("OME channel count:", len(full_merge_info["ome_channel_names"]))
print("Channel aliases match OME names:", full_merge_info["ome_channel_names"] == channel_aliases)
print("Physical size X/Y:", full_merge_info["physical_size_x"], full_merge_info["physical_size_y"])


In [ ]:
def to_cyx(array, axes: str):
    axes = axes.upper()
    if axes == "YX":
        return np.expand_dims(array, axis=0)
    if set(axes) != set("CYX"):
        raise ValueError(f"Unsupported axes for manual import: {axes!r}")
    order = [axes.index(axis) for axis in "CYX"]
    return np.transpose(array, order)


def get_scale0_image(image_element):
    if isinstance(image_element, DataTree):
        return image_element["scale0"]["image"]
    return image_element


def extract_channel_names(image_element) -> list[str]:
    base_image = get_scale0_image(image_element)
    try:
        coords = base_image.coords
    except Exception:
        return []
    if "c" not in coords:
        return []
    values = coords["c"].values.tolist()
    return [str(value) for value in values]


def summarize_image_element(image_element) -> dict:
    base_image = get_scale0_image(image_element)
    dims = tuple(str(dim) for dim in getattr(base_image, "dims", ()))
    shape = tuple(int(size) for size in getattr(base_image, "shape", ()))
    return {
        "element_type": type(image_element).__name__,
        "is_multiscale": isinstance(image_element, DataTree),
        "dims": dims,
        "shape": shape,
        "channel_names": extract_channel_names(image_element),
    }


def manual_image_import(path: Path):
    with tifffile.TiffFile(path) as handle:
        series = handle.series[0]
        array = series.asarray(level=0)
        array = to_cyx(array, series.axes)
    return Image2DModel.parse(array, dims=("c", "y", "x"), c_coords=channel_aliases)


image_options: dict[str, callable] = {
    "sopa.io.ome_tif": lambda: sopa.io.ome_tif(FULL_MERGE_PATH),
    "spatialdata_io.image": lambda: spatialdata_io.image(
        FULL_MERGE_PATH,
        data_axes=("c", "y", "x"),
        coordinate_system="global",
    ),
    "spatialdata_io.generic": lambda: spatialdata_io.generic(
        FULL_MERGE_PATH,
        data_axes=("c", "y", "x"),
        coordinate_system="global",
    ),
    "manual Image2DModel.parse": lambda: manual_image_import(FULL_MERGE_PATH),
}

image_results = []
image_payloads = {}

for option_name, loader in image_options.items():
    try:
        payload = loader()
        if isinstance(payload, SpatialData):
            image_key = next(iter(payload.images))
            image_element = payload.images[image_key]
            payload_kind = "SpatialData"
        else:
            image_key = "image"
            image_element = payload
            payload_kind = type(payload).__name__

        summary = summarize_image_element(image_element)
        summary.update(
            {
                "option": option_name,
                "success": True,
                "payload_kind": payload_kind,
                "image_key": image_key,
                "channel_names_match_aliases": summary["channel_names"] == channel_aliases,
                "channel_count_matches": len(summary["channel_names"]) == len(channel_aliases),
                "error": "",
            }
        )
        image_results.append(summary)
        image_payloads[option_name] = {
            "payload": payload,
            "image_element": image_element,
            "image_key": image_key,
        }
    except Exception as exc:
        image_results.append(
            {
                "option": option_name,
                "success": False,
                "payload_kind": "",
                "image_key": "",
                "element_type": "",
                "dims": (),
                "shape": (),
                "is_multiscale": False,
                "channel_names": [],
                "channel_names_match_aliases": False,
                "channel_count_matches": False,
                "error": repr(exc),
            }
        )

comparison_df = pd.DataFrame(image_results)
comparison_df


In [ ]:
def preferred_option(results_df: pd.DataFrame) -> str:
    successful = results_df[results_df["success"]].copy()
    if successful.empty:
        raise RuntimeError("No image import option succeeded.")

    preference_order = {
        "sopa.io.ome_tif": 0,
        "spatialdata_io.image": 1,
        "spatialdata_io.generic": 2,
        "manual Image2DModel.parse": 3,
    }

    successful["score"] = (
        successful["channel_names_match_aliases"].astype(int) * 10
        + successful["channel_count_matches"].astype(int) * 3
        + successful["is_multiscale"].astype(int) * 2
    )
    successful["tie_breaker"] = successful["option"].map(preference_order).fillna(999)
    successful = successful.sort_values(["score", "tie_breaker"], ascending=[False, True])
    return str(successful.iloc[0]["option"])


PREFERRED_IMAGE_OPTION = preferred_option(comparison_df)
print("Preferred image import option:", PREFERRED_IMAGE_OPTION)

selected_image = image_payloads[PREFERRED_IMAGE_OPTION]["image_element"]
canonical_sdata = SpatialData(images={"full_image": selected_image})

selected_summary = comparison_df.loc[comparison_df["option"] == PREFERRED_IMAGE_OPTION].iloc[0]
selected_summary


### Notes on choosing the preferred image import route

Use the comparison table above to decide what carries forward into the full-slide implementation.

Default preference order in this notebook:
1. exact channel-name preservation,
2. matching channel count,
3. multiscale image support,
4. simpler high-level reader before manual construction.

In the current `spatialdata` conda env used during development, `sopa.io.ome_tif(...)` preserved the aliases and multiscale pyramid cleanly, while `spatialdata_io.image(...)` and `spatialdata_io.generic(...)` loaded successfully but replaced channel coordinates with numeric strings.

If the automatically selected option is not the one you want, set `PREFERRED_IMAGE_OPTION` manually in the cell above and rerun from that point.


In [ ]:
def inspect_mask(path: Path) -> dict:
    array = tifffile.imread(path)
    nonzero = array[array > 0]
    return {
        "path": str(path),
        "shape": tuple(int(v) for v in array.shape),
        "dtype": str(array.dtype),
        "min": int(array.min()),
        "max": int(array.max()),
        "nonzero_pixel_count": int(nonzero.size),
        "unique_nonzero_labels": int(np.unique(nonzero).size) if nonzero.size else 0,
    }, array


cell_mask_info, cell_mask_array = inspect_mask(CELL_MASK_PATH)
cell_mask_info


In [ ]:
selected_scale0_image = get_scale0_image(selected_image)
image_shape_yx = tuple(int(v) for v in selected_scale0_image.shape[-2:])
cell_mask_shape_yx = tuple(int(v) for v in cell_mask_array.shape[-2:])

print("Image YX shape:", image_shape_yx)
print("Cell mask YX shape:", cell_mask_shape_yx)

if image_shape_yx != cell_mask_shape_yx:
    raise ValueError(
        f"Cell mask shape {cell_mask_shape_yx} does not match image shape {image_shape_yx}."
    )

cell_labels = Labels2DModel.parse(cell_mask_array, dims=("y", "x"))
labels_dict = {"cell_labels": cell_labels}

if NUCLEAR_MASK_PATH.exists():
    nuclear_mask_info, nuclear_mask_array = inspect_mask(NUCLEAR_MASK_PATH)
    if tuple(int(v) for v in nuclear_mask_array.shape[-2:]) != image_shape_yx:
        raise ValueError(
            f"Nuclear mask shape {nuclear_mask_array.shape} does not match image shape {image_shape_yx}."
        )
    labels_dict["nuclear_labels"] = Labels2DModel.parse(nuclear_mask_array, dims=("y", "x"))
    print("Loaded nuclear mask as labels.")
else:
    nuclear_mask_info = None
    nuclear_mask_array = None
    print("No nuclear mask found; skipping nuclear label import.")

canonical_sdata = SpatialData(images={"full_image": selected_image}, labels=labels_dict)
canonical_sdata


In [ ]:
alignment_summary = {
    "image_element_type": type(canonical_sdata.images["full_image"]).__name__,
    "image_dims": tuple(str(dim) for dim in get_scale0_image(canonical_sdata.images["full_image"]).dims),
    "image_shape": tuple(int(v) for v in get_scale0_image(canonical_sdata.images["full_image"]).shape),
    "cell_label_dims": tuple(str(dim) for dim in canonical_sdata.labels["cell_labels"].dims),
    "cell_label_shape": tuple(int(v) for v in canonical_sdata.labels["cell_labels"].shape),
    "same_canvas": tuple(int(v) for v in get_scale0_image(canonical_sdata.images["full_image"]).shape[-2:])
    == tuple(int(v) for v in canonical_sdata.labels["cell_labels"].shape[-2:]),
    "coordinate_systems": list(getattr(canonical_sdata, "coordinate_systems", [])),
}

alignment_summary


In [ ]:
if HAS_MATPLOTLIB:
    image_array = np.asarray(get_scale0_image(canonical_sdata.images["full_image"]))
    label_array = np.asarray(canonical_sdata.labels["cell_labels"])

    channel_index = 0
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
    axes[0].imshow(image_array[channel_index], cmap="gray")
    axes[0].set_title(f"full_image channel {channel_index}: {channel_aliases[channel_index]}")
    axes[1].imshow(label_array, cmap="tab20")
    axes[1].set_title("cell_labels")
    axes[2].imshow(image_array[channel_index], cmap="gray")
    axes[2].contour(label_array > 0, levels=[0.5], colors="red", linewidths=0.5)
    axes[2].set_title("overlay")
    for ax in axes:
        ax.set_axis_off()
    plt.show()
else:
    print("matplotlib not available; skipping overlay plot.")


## Optional Next Experiment: polygonize `cell_labels`

Only run this after the raster-label workflow is working.

Why defer this:
- the current prototype goal is raster ingestion and alignment,
- polygonization introduces extra dependencies (`geopandas`, `shapely`, often `rasterio`),
- once labels are stable, we can test `ShapesModel.parse(...)` as the next step toward Sopa aggregation.


In [ ]:
RUN_OPTIONAL_POLYGONIZE = False

if RUN_OPTIONAL_POLYGONIZE:
    geopandas = require_module("geopandas")
    rasterio_features = import_module("rasterio.features")
    shapely_geometry = import_module("shapely.geometry")
    from spatialdata.models import ShapesModel

    polygons = []
    for geometry_mapping, value in rasterio_features.shapes(cell_mask_array, mask=cell_mask_array > 0):
        polygons.append(
            {
                "label_id": str(int(value)),
                "geometry": shapely_geometry.shape(geometry_mapping),
            }
        )

    gdf = geopandas.GeoDataFrame(polygons).set_index("label_id")
    cell_boundaries = ShapesModel.parse(gdf)
    canonical_sdata = SpatialData(
        images={"full_image": canonical_sdata.images["full_image"]},
        labels=canonical_sdata.labels,
        shapes={"cell_boundaries": cell_boundaries},
    )
    canonical_sdata
else:
    print("Set RUN_OPTIONAL_POLYGONIZE = True after raster import is validated.")


## References

- SpatialData: https://spatialdata.scverse.org/en/stable/index.html
- SpatialData models: https://spatialdata.scverse.org/en/stable/api/models.html
- spatialdata-io generic: https://spatialdata.scverse.org/projects/io/en/stable/generated/spatialdata_io.generic.html
- Sopa readers: https://gustaveroussy.github.io/sopa/api/readers/
